# 더티 데이터 실습: 지저분한 iris 청소하기 (30분)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/slides/v2/ml1/week01/dirty_data/lab/dirty_data.ipynb)

붓꽃(iris) 150송이 데이터에 **일부러** 더러운 값을 섞어 두었다. 슬라이드의 세 가지 모양(결측치 · 중복 · 이상치)을 찾아서 고치고, 고치기 전과 후에 모델 정확도가 얼마나 다른지 본다.

| 파트 | 내용 | 시간 |
|---|---|---|
| 0 | 더러운 데이터 받기, 첫눈에 훑어보기 | 4분 |
| 1 | 결측치: 빈칸 · 0 · -999 찾기 | 7분 |
| 2 | 중복과 라벨 표기: `'Setosa'`, `' setosa'` 는 같은 꽃? | 7분 |
| 3 | 이상치: 51cm 꽃받침은 오타일까? | 6분 |
| 4 | 청소 전 vs 후: 같은 모델의 정확도 비교 | 6분 |

`None` 으로 비워 둔 `TODO` 줄을 채우고 셀을 차례로 실행하세요.

In [ ]:
import os, urllib.request
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm
_have = {f.name for f in fm.fontManager.ttflist}
_font = next((f for f in ["Noto Sans CJK KR", "NanumGothic", "AppleGothic", "Malgun Gothic"] if f in _have), None)
if _font is None:   # Colab 등 한글 글꼴이 없으면 나눔고딕을 받아 등록
    try:
        _p = "/tmp/NanumGothic.ttf"
        if not os.path.exists(_p):
            urllib.request.urlretrieve("https://github.com/google/fonts/raw/main/ofl/nanumgothic/NanumGothic-Regular.ttf", _p)
        fm.fontManager.addfont(_p); _font = fm.FontProperties(fname=_p).get_name()
    except Exception as e:
        print("한글 글꼴 설치 실패 (그래프 글자만 깨지고 실습은 가능):", e)
if _font:
    plt.rcParams["font.family"] = _font
plt.rcParams["axes.unicode_minus"] = False
import pandas as pd
from sklearn.datasets import load_iris

# ---- 일부러 더럽히기 (이 셀은 고치지 말고 실행만) ----
iris = load_iris(as_frame=True)
clean = iris.frame.rename(columns={"sepal length (cm)": "sepal_length", "sepal width (cm)": "sepal_width",
                                   "petal length (cm)": "petal_length", "petal width (cm)": "petal_width"})
clean["species"] = iris.target_names[iris.target]
clean = clean.drop(columns="target")

def make_dirty(seed=2026):
    """깨끗한 iris 에 결측치 · 중복 · 이상치 · 표기 흔들림을 섞는다"""
    rng = np.random.default_rng(seed)
    d = clean.copy()
    d.loc[rng.choice(150, 5, replace=False), "petal_length"] = np.nan           # 진짜 빈칸
    d.loc[rng.choice(150, 4, replace=False), "sepal_width"] = -999              # '비었음'을 -999 로
    d.loc[rng.choice(np.where(clean.species != "setosa")[0], 5, replace=False), "petal_width"] = 0   # '비었음'을 0 으로
    d.loc[rng.choice(150, 3, replace=False), "sepal_length"] *= 10             # cm 대신 mm 로 적은 오타
    styles = [str.capitalize, lambda s: " " + s, lambda s: s + " ", str.upper]
    for i, f in zip(rng.choice(150, 4, replace=False), styles):              # 표기만 다른 라벨
        d.loc[i, "species"] = f(d.loc[i, "species"])
    dups = d.loc[rng.choice(150, 8, replace=False)].copy()                    # 같은 기록이 두 번
    dups.iloc[:3, dups.columns.get_loc("species")] = [str.upper(v) for v in dups.species.iloc[:3]]  # 그중 3개는 표기만 다르게
    return pd.concat([d, dups]).sample(frac=1, random_state=0).reset_index(drop=True)

df = make_dirty()
print(df.shape); df.head()

## 파트 0. 첫눈에 훑어보기

`describe()` 로 각 열의 최솟값 · 최댓값 · 평균을 본다. 이상한 점이 몇 개 보이나?

In [ ]:
df.describe().round(2)

## 파트 1. 결측치: 빈칸 · 0 · -999

빈칸(`NaN`)은 `isna()` 가 찾아 주지만, `0` 과 `-999` 는 숫자인 척한다. 꽃받침·꽃잎 길이가 0cm 이하일 수 있을까?

In [ ]:
num_cols = ["sepal_length", "sepal_width", "petal_length", "petal_width"]
print("빈칸(NaN) 개수:\n", df[num_cols].isna().sum(), "\n")
print("0 이하인 값 개수:\n", (df[num_cols] <= 0).sum())

In [ ]:
# 숫자인 척하는 '비었음' (0 이하) 을 진짜 빈칸 NaN 으로 바꾼다
df[num_cols] = None   # TODO: 0 이하 → NaN (힌트: .mask)
print(df[num_cols].isna().sum())

## 파트 2. 중복과 라벨 표기

`value_counts()` 로 꽃 종류를 세어 보자. 종류가 3개여야 하는데 몇 개인가?

In [ ]:
print(df["species"].value_counts(), "\n")
print("완전히 같은 줄:", df.duplicated().sum(), "개")

In [ ]:
# 표기를 하나로: 앞뒤 공백 제거 + 소문자
df["species"] = None    # TODO: 공백 제거 + 소문자
print(df["species"].value_counts(), "\n")
print("표기를 맞춘 뒤 같은 줄:", df.duplicated().sum(), "개")
df = df.drop_duplicates().reset_index(drop=True)
print("중복 지운 뒤:", df.shape)

**관찰**: 표기를 맞추기 전에는 컴퓨터가 찾은 중복 수가 실제보다 적다. 띄어쓰기나 대소문자 하나만 달라도 다른 줄로 본다. 무엇을 같은 것으로 볼지는 사람이 정한다.

## 파트 3. 이상치: 51cm 꽃받침?

붓꽃의 꽃받침 길이는 보통 4~8cm 다. 상자 그림으로 튀는 값을 찾아보자.

In [ ]:
df[num_cols].plot(kind="box", figsize=(8, 3.5)); plt.ylabel("cm"); plt.show()
print(df.loc[df.sepal_length > 10, ["sepal_length", "species"]])

In [ ]:
# 분야 지식: 꽃받침이 10cm 를 넘는 값은 mm 로 적은 오타 → 10 으로 나눠 cm 로
too_big = df.sepal_length > 10
df.loc[too_big, "sepal_length"] = None   # TODO: mm → cm
print(df.sepal_length.describe().round(2))

**생각해 보기**: 여기서는 '10 으로 나누기' 로 고쳤다. 만약 꽃받침이 9.5cm 라면? 오타인지 진짜 큰 꽃인지는 데이터만 봐서는 모른다. 지울지, 고칠지, 남길지는 그 분야를 아는 사람이 정한다.

## 파트 4. 청소 전 vs 후: 모델은 얼마나 달라질까?

남은 빈칸은 그 꽃 종류의 평균으로 채우고 (꽃 종류별 평균), 같은 모델(로지스틱회귀, 5-겹 교차검증) 로 정확도를 비교한다. 비교용 '청소 안 한' 데이터는 첫 셀의 `make_dirty()` 결과에서 빈칸만 0 으로 채운 것이다.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# 청소한 데이터: 남은 빈칸을 꽃 종류별 평균으로
filled = df.copy()
filled[num_cols] = None   # TODO: 종류별 평균으로 채우기

# 청소 안 한 데이터: 첫 셀과 같은 make_dirty() 에서 빈칸만 0 으로
raw = make_dirty().fillna(0)

import warnings; warnings.filterwarnings("ignore")      # 1개짜리 '종류' 때문에 나는 경고는 숨긴다
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
for name, data in [("청소 안 함", raw), ("청소함", filled), ("원래 깨끗한 iris", clean)]:
    acc = cross_val_score(model, data[num_cols], data["species"], cv=5)
    print(f"{name:14s}: 꽃 종류 {data.species.nunique()}개, 정확도 {acc.mean():.1%}")

**확인 질문**
1. 파트 1 에서 `-999` 를 그대로 두고 꽃받침 너비의 평균을 내면 얼마가 되나? 실제 평균(약 3cm)과 비교해 보자.
2. 파트 2 에서 표기를 맞추기 전과 후에 `duplicated()` 가 찾은 수가 다른 이유는?
3. 파트 3 에서 꽃받침 9.5cm 는 오타일까 진짜일까? 무엇을 더 알아야 판단할 수 있을까?
4. 파트 4 에서 '청소 안 함' 의 꽃 종류가 3개가 아닌 이유는? 모델은 무엇을 배우게 될까?

데이터 출처: Fisher (1936), Iris (scikit-learn 내장).